In [1]:
%load_ext autoreload
%autoreload 2

# LwF with trainable backbone

This notebook implements Learning without Forgetting from scratch with a switch for TIL or CIL. The backbone stays trainable, and the notebook does not use the repository's `LwFCriterion` implementation.

In [2]:
import copy
import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
for extra_path in (ROOT, ROOT / "models", ROOT / "src"):
    extra = str(extra_path)
    if extra not in sys.path:
        sys.path.append(extra)

from models.backbone import BackBone
from models.multiheadmodel import MultiHeadModel
from src.dataset import get_data_loaders
from src.utils import deterministic, accuracy


EXPERIMENT_MODE = "CIL"  # Change to "TIL" to test task-incremental learning.
CLASSES_PER_TASK = 2
LWF_ALPHA = 1.0
LWF_TEMPERATURE = 2.0
EPOCHS_PER_TASK = 5
BATCH_SIZE = 128


def make_teacher(model):
    teacher = copy.deepcopy(model)
    teacher.eval()
    for parameter in teacher.parameters():
        parameter.requires_grad = False
    return teacher


def classification_targets(y):
    return y


def distillation_loss(student, teacher, x, task_number, global_labels, temperature=2.0):
    if teacher is None:
        return torch.tensor(0.0, device=x.device)

    if global_labels:
        with torch.no_grad():
            teacher_logits = teacher(x, 0)
        student_logits = student(x, 0)[:, : teacher_logits.size(1)]
        student_log_probs = F.log_softmax(student_logits / temperature, dim=1)
        teacher_probs = F.softmax(teacher_logits / temperature, dim=1)
        return F.kl_div(student_log_probs, teacher_probs, reduction="batchmean") * (temperature ** 2)

    old_tasks = max(0, task_number)
    if old_tasks == 0:
        return torch.tensor(0.0, device=x.device)

    distill = 0.0
    for old_task in range(old_tasks):
        with torch.no_grad():
            teacher_logits = teacher(x, old_task)
        student_logits = student(x, old_task)
        student_log_probs = F.log_softmax(student_logits / temperature, dim=1)
        teacher_probs = F.softmax(teacher_logits / temperature, dim=1)
        distill = distill + F.kl_div(student_log_probs, teacher_probs, reduction="batchmean")

    return (distill / old_tasks) * (temperature ** 2)


def train_lwf_task(
    model,
    train_loader,
    optimizer,
    criterion,
    task_number,
    epochs=5,
    alpha=1.0,
    temperature=2.0,
    global_labels=False,
    teacher=None,
):
    device = next(model.parameters()).device
    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        progress = tqdm(train_loader, desc=f"Task {task_number} | Epoch {epoch + 1}/{epochs}", leave=False)
        for x, y in progress:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            head_number = 0 if global_labels else task_number
            targets = classification_targets(y)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x, head_number)
            loss = criterion(logits, targets)

            if teacher is not None:
                loss = loss + alpha * distillation_loss(
                    model,
                    teacher,
                    x,
                    task_number,
                    global_labels,
                    temperature=temperature,
                )

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            progress.set_postfix(loss=loss.item())

        print(f"Task {task_number} epoch {epoch + 1}: loss={running_loss / max(1, len(train_loader)):.4f}")


def evaluate_seen_tasks(model, dataloaders, upto_task, global_labels):
    task_accuracies = []
    for eval_task in range(upto_task + 1):
        eval_loader = dataloaders[eval_task][2]
        eval_head = 0 if global_labels else eval_task
        eval_acc = accuracy(model, eval_loader, eval_head, global_labels=global_labels)
        task_accuracies.append(eval_acc)
        print(f"  Task {eval_task}: test_acc={eval_acc:.4f}")
    return sum(task_accuracies) / len(task_accuracies)


def run_lwf_experiment(mode="CIL", alpha=1.0, temperature=2.0, seed=42, epochs=5, batch_size=128):
    deterministic(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dataloaders = get_data_loaders(val_size=0.1, batch_size=batch_size)

    model = MultiHeadModel(BackBone()).to(device)
    for parameter in model.backbone.parameters():
        parameter.requires_grad = True

    criterion = nn.CrossEntropyLoss()
    global_labels = mode.upper() == "CIL"
    teacher = None
    seen_task_accs = []

    for task_number, (train_loader, val_loader, test_loader) in enumerate(dataloaders):
        if global_labels:
            if task_number == 0:
                model.add_head(0, num_classes=CLASSES_PER_TASK)
            else:
                model.expand_head(0, num_class_increment=CLASSES_PER_TASK)
            head_number = 0
        else:
            model.add_head(task_number, num_classes=CLASSES_PER_TASK)
            head_number = task_number

        model.to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

        train_lwf_task(
            model=model,
            train_loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            task_number=head_number,
            epochs=epochs,
            alpha=alpha,
            temperature=temperature,
            global_labels=global_labels,
            teacher=teacher,
        )

        print(f"\nTask {task_number} evaluation on all seen tasks ({mode.upper()})")
        avg_seen_acc = evaluate_seen_tasks(model, dataloaders, task_number, global_labels)
        seen_task_accs.append(avg_seen_acc)

        teacher = make_teacher(model)

    return {
        "mode": mode.upper(),
        "alpha": alpha,
        "temperature": temperature,
        "seen_task_avg_history": seen_task_accs,
        "final_seen_task_avg": seen_task_accs[-1],
    }

In [3]:
MODE_TO_RUN = EXPERIMENT_MODE.upper()
result = run_lwf_experiment(
    mode=MODE_TO_RUN,
    alpha=LWF_ALPHA,
    temperature=LWF_TEMPERATURE,
    seed=42,
    epochs=EPOCHS_PER_TASK,
    batch_size=BATCH_SIZE,
)
print("\nSummary:")
print(
    f"mode={result['mode']}, alpha={result['alpha']}, temperature={result['temperature']} -> "
    f"final_seen_task_avg={result['final_seen_task_avg']:.4f}"
)
print("\nInterpretation:")
print("- The backbone is trainable again, so this matches the earlier TIL/CIL notebook structure.")
print("- CIL still uses one expanding head with distillation; TIL uses separate heads.")
print("- If forgetting is still strong, LwF alone may not be enough for your CIL setting.")

/home/alumno1/miniconda3/envs/vision/lib/python3.11/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Task [0, 1]: Train=9000, Val=1000, Test=2000
Task [2, 3]: Train=9000, Val=1000, Test=2000
Task [4, 5]: Train=9000, Val=1000, Test=2000
Task [6, 7]: Train=9000, Val=1000, Test=2000
Task [8, 9]: Train=9000, Val=1000, Test=2000


Task 0 | Epoch 1/5:   0%|          | 0/71 [00:00<?, ?it/s]

Task 0 epoch 1: loss=0.3346


Task 0 | Epoch 2/5:   0%|          | 0/71 [00:00<?, ?it/s]

...

Task 0 epoch 3: loss=0.2473


Task 0 | Epoch 4/5:   0%|          | 0/71 [00:00<?, ?it/s]

Task 0 epoch 4: loss=0.1270


Task 0 | Epoch 5/5:   0%|          | 0/71 [00:00<?, ?it/s]

Task 0 epoch 5: loss=0.0854

Task 4 evaluation on all seen tasks (CIL)
  Task 0: test_acc=0.0000
  Task 1: test_acc=0.0000
  Task 2: test_acc=0.0000
  Task 1: test_acc=0.0000
  Task 2: test_acc=0.0000
  Task 3: test_acc=0.0030
  Task 4: test_acc=0.9135

Summary:
mode=CIL, alpha=1.0, temperature=2.0 -> final_seen_task_avg=0.1833

Interpretation:
- The backbone is trainable again, so this matches the earlier TIL/CIL notebook structure.
- CIL still uses one expanding head with distillation; TIL uses separate heads.
- If forgetting is still strong, LwF alone may not be enough for your CIL setting.
  Task 3: test_acc=0.0030
  Task 4: test_acc=0.9135

Summary:
mode=CIL, alpha=1.0, temperature=2.0 -> final_seen_task_avg=0.1833

Interpretation:
- The backbone is trainable again, so this matches the earlier TIL/CIL notebook structure.
- CIL still uses one expanding head with distillation; TIL uses separate heads.
- If forgetting is still strong, LwF alone may not be enough for your CIL setting.